# Blueprint to pipeline to three substrates: one backend-neutral blueprint, run faithfully on CPM, vertex and center-based, with a pluggable subcellular layer

_Investigation `influenza-crossparadigm` — coder reproduction notebook._

**Question.** Can ONE versioned, backend-neutral blueprint be translated by a single pipeline (`blueprint_pipeline`) and run faithfully on three different mechanical substrates -- a Cellular Potts model (viva-cpm), a vertex model (vivarium-tyssue) and a center-based model (viva-munk) -- including a pluggable per-cell subcellular layer (a tellurium ODE that grows and divides cells); and where do the substrates agree vs diverge when the shared mechanism is driven live? The Sego 2022 influenza model (ViralInfectionVTM) is the worked example for the comparison; a v2 subcellular growth/division blueprint is the worked example for the subcellular layer.

A demonstration that one backend-neutral blueprint, compiled by the single `blueprint_pipeline`, runs the Sego 2022 influenza model and a per-cell subcellular ODE model on a Cellular Potts, a vertex and a center-based substrate, with a full comparison and interactive figures.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


**Competing frameworks**

- **Cellular Potts (CPM / viva-cpm)**
- **Vertex model (tyssue / vivatyssue)**
- **Center-based model (viva-munk)**
- **Framing**


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint').is_dir():
    REPO = Path('/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_multicell_blueprint.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: One blueprint, one pipeline: contract and t0 fidelity (`blueprint-pipeline-fidelity`)

**Question.** Can one versioned, backend-neutral blueprint be compiled by `blueprint_pipeline` and materialized faithfully on CPM, vertex and center-based?

**Purpose.** A canonical schema + a split that routes backend/run parameters to CompileOptions keeps the biological state immutable across backends, and a static validator turns each semantic invariant into a checked diagnostic. CPM then materializes each blueprint cell 1:1 at its centroid with target_volume = size; the vertex backend assigns cells 1:1 to a regular hexagonal lattice of the target area (alternating optimal assignment with rigid recentring) and attaches the latent immune cells at the tissue-margin site nearest their centroid; the center-based backend places one area-matched disc per cell.

**Claim.** ONE versioned, backend-neutral blueprint is compiled by the single `blueprint_pipeline` composite and materialized faithfully on the CPM, vertex and center-based backends. Design/Build half: the blueprint is a first-class artifact with a versioned schema, a canonical/backend split (backend and run parameters live in CompileOptions, never in the biological state) and a static validator with structured diagnostics. Simulate/ Evaluate half: the t0 state each backend actually emits matches the blueprint: 48/48 cell types, CPM centroid RMS 0.000 um, and a vertex sheet of 48 hexagons (36 epithelial + 12 immune attached at the tissue margin) with epithelial centroid RMS 1.60 um (max 2.44 um) and immune attachment RMS 4.11 um (max 5.60 um).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `sego-t0-fidelity` | `viva_multicell_blueprint.composites.pipeline.blueprint_pipeline` | 0 | backend=cpm, blueprint=workspace/blueprints/influenza_sego2022.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.pipeline.blueprint_pipeline`**

Backend-neutral blueprint compiled onto a mechanics backend (cpm|vertex|munk): fields/rules/ODE/immune mechanism, plus an optional per-cell subcellular layer arbitrated into the mechanics inputs by the actuation merge.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm', 'vertex', or 'munk'
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # a path to a YAML file, inline YAML text, or a dict (v1 flat or v2 sectioned)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: blueprint-pipeline-fidelity ===
STUDY = 'blueprint-pipeline-fidelity'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_sego_t0_fidelity = 100
INTERVAL_sego_t0_fidelity = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline if 'ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline = materialize_composite('viva_multicell_blueprint.composites.pipeline.blueprint_pipeline', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline['state']}, core=core)
            _comp.run(STEPS_sego_t0_fidelity)
        print('ran', 'sego-t0-fidelity', f'(viva_multicell_blueprint.composites.pipeline.blueprint_pipeline) for', STEPS_sego_t0_fidelity, 'steps')
    except Exception as _e:
        print(f"sego-t0-fidelity not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**t0 centroid residuals**


In [ ]:
# t0 centroid residuals
show_viz(_render_one('html:t0-residuals.html', {'title': 'Per-cell t0 centroid residual per backend'}, RUNS_DB, STUDY_YAML))

**t0 side-by-side**


In [ ]:
# t0 side-by-side
show_viz(_render_one('html:t0-side-by-side.html', {'title': 'Initial state: CPM | vertex | center-based'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| same-canonical-blueprint-all-backends _(primary)_ | kind=contract field=canonical_state | op both_backends_accept note tests/test_blueprint_canonical.py::test_same_canonical_blueprint_compiles_to_both_backends |
| backend-run-params-not-canonical _(primary)_ | kind=contract field=canonical_violations | op separated_and_flagged note tests/test_blueprint_canonical.py::test_backend_run_params_are_not_canonical |
| validator-structured-diagnostics _(primary)_ | kind=contract field=validate_blueprint | op diagnostics_shape note tests/test_blueprint_canonical.py::test_validator_structured_diagnostics_and_violation_warning |
| cpm-exact-t0 _(primary)_ | kind=residual field=centroid_residual backend=cpm | op lt value 1e-06 field centroid_rms |
| vertex-clean-identity _(primary)_ | kind=residual field=centroid_residual backend=vertex | op lt value 2.0 field sheet_centroid_rms note fraction_clean == 1.0; n_guard_faces == 0 |
| vertex-one-hexagonal-sheet _(primary)_ | kind=topology field=vertex_residual_report | op eq value 48 field n_sheet note n_overlay == 0; n_attached_immune == 12; immune_attach_max < 6 um |
| all-types-match _(primary)_ | kind=count field=type_match | op eq value 48 field n_type_match |


## Study: Cross-paradigm equivalence: fate yes, exact position no, and the test has power (`cross-paradigm-equivalence`)

**Question.** Under replication, are CPM, vertex and center-based EQUIVALENT (not merely "not-distinguishable") when one blueprint is run through `blueprint_pipeline` -- on cell fate, on exact spatial position, or both -- and does the test correctly reject when mechanics dominates?

**Purpose.** The epithelial rates, four reaction-diffusion fields, Price-2015 systemic ODE (fed live by CellCountStep) and latent-pool immune recruitment are compiled identically onto every backend; only the single mechanics node differs (CPM Metropolis flips, vertex tyssue relaxation, center-based hard-disc relaxation). Fate is driven by the shared mechanism, so it should be backend-neutral; exact position is set by each backend's own mechanics and should drift apart. If the substrate mattered beyond noise, cross-backend metrics would fall OUTSIDE the same-backend seed spread; in the mechanics-only control nothing shared forces agreement, so the test must reject.

**Claim.** Run through `blueprint_pipeline` on CPM, vertex and center-based (viva-munk), the ONE Sego 2022 blueprint gives backend-neutral cell FATE and backend-specific exact POSITION, and the equivalence test is powerful enough to tell them apart. FATE agreement is statistically EQUIVALENT for every backend pair within a pre-declared +/-0.05 margin (run-cluster 90% CI [-0.0034, +0.0008] for cpm-vertex). Exact-position agreement is NOT equivalent for any pair at +/-0.10, and it orders by how different the geometries are: cpm-munk +0.093 < cpm-vertex +0.126 < vertex-munk +0.252 (the cpm-munk point estimate sits below the 0.10 margin but its CI [0.034, 0.159] does not fit inside +/-0.10, so equivalence still cannot be claimed). A mechanics-dominated positive-control arm REJECTS equivalence for every pair with the same test (cpm-vertex Delta +0.325), so a "pass" on fate is informative. Judged against the same-backend seed null, the N=400 capstone fate metric fails to reject a substrate difference, while infected-patch occupancy IoU and patch location now sit outside both same-backend baselines for the current vertex substrate: a candidate substrate signal, not causally attributed.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `sego-equivalence-grid` | `viva_multicell_blueprint.composites.pipeline.blueprint_pipeline` | 0 | backend=cpm, blueprint=workspace/blueprints/influenza_sego2022.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.pipeline.blueprint_pipeline`**

Backend-neutral blueprint compiled onto a mechanics backend (cpm|vertex|munk): fields/rules/ODE/immune mechanism, plus an optional per-cell subcellular layer arbitrated into the mechanics inputs by the actuation merge.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm', 'vertex', or 'munk'
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # a path to a YAML file, inline YAML text, or a dict (v1 flat or v2 sectioned)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: cross-paradigm-equivalence ===
STUDY = 'cross-paradigm-equivalence'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_sego_equivalence_grid = 100
INTERVAL_sego_equivalence_grid = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline if 'ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline = materialize_composite('viva_multicell_blueprint.composites.pipeline.blueprint_pipeline', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline['state']}, core=core)
            _comp.run(STEPS_sego_equivalence_grid)
        print('ran', 'sego-equivalence-grid', f'(viva_multicell_blueprint.composites.pipeline.blueprint_pipeline) for', STEPS_sego_equivalence_grid, 'steps')
    except Exception as _e:
        print(f"sego-equivalence-grid not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Fate equivalence (TOST)**


In [ ]:
# Fate equivalence (TOST)
show_viz(_render_one('html:equivalence-fate.html', {'title': 'Cross- vs same-backend fate agreement with the equivalence margin'}, RUNS_DB, STUDY_YAML))

**Spatial (exact-position) equivalence (TOST)**


In [ ]:
# Spatial (exact-position) equivalence (TOST)
show_viz(_render_one('html:equivalence-spatial.html', {'title': 'Cross- vs same-backend exact-position agreement'}, RUNS_DB, STUDY_YAML))

**Spatial playback -- CPM vs vertex vs center-based**


In [ ]:
# Spatial playback -- CPM vs vertex vs center-based
show_viz(_render_one('html:spatial-playback.html', {'title': 'Spatial playback: CPM | vertex | center-based'}, RUNS_DB, STUDY_YAML))

**Dynamics overlay (I-count band)**


In [ ]:
# Dynamics overlay (I-count band)
show_viz(_render_one('html:dynamics-overlay.html', {'title': 'Dynamics vs the CPM stochastic-envelope band'}, RUNS_DB, STUDY_YAML))

**Divergence map**


In [ ]:
# Divergence map
show_viz(_render_one('html:divergence.html', {'title': 'Per-cell agreement + spatial map (vs same-backend baseline)'}, RUNS_DB, STUDY_YAML))

**Positive control -- spatial equivalence REJECTED**


In [ ]:
# Positive control -- spatial equivalence REJECTED
show_viz(_render_one('html:counterfactual-equivalence-spatial.html', {'title': 'Mechanics-dominated regime: cross- vs same-backend spatial agreement'}, RUNS_DB, STUDY_YAML))

**Positive control -- mechanics-dominated playback**


In [ ]:
# Positive control -- mechanics-dominated playback
show_viz(_render_one('html:counterfactual-spatial-playback.html', {'title': 'Divergent sorting: CPM voxel | vertex polygon | center-based'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| fate-equivalent-tost _(primary)_ | kind=equivalence field=fate_agreement margin=0.05 | op tost_equivalent note 90% CI of (same-cross) inside +/-0.05 |
| spatial-equivalence-rejected-in-mechanics-regime _(primary)_ | kind=equivalence field=spatial_agreement margin=0.1 | op tost_not_equivalent note cross spatial agreement << same; diff > margin |
| both-backends-run-end-to-end _(primary)_ | kind=count field=frames when=end-of-run | op eq value 151 field n_frames_at_N150 |
| fate-agreement-within-seed-baseline _(primary)_ | kind=delta field=fate_agreement compare_to=same_backend_cpm_seed_baseline | op gte value -0.05 field cross_minus_baseline |
| spatial-not-representation-confounded _(primary)_ | kind=ratio field=final_mean_area versus=cpm | op in_range lo 0.6 hi 1.6 field area_ratio |
| spatial-gap-is-dynamic-not-offset _(secondary)_ | kind=comparison field=spatial_agreement low=0.0 high=1.0 | op strictly_greater note cross spatial(t0) > cross spatial(final) |
| vertex-within-cpm-envelope _(secondary)_ | kind=fraction field=inband | op gte value 0.8 field vertex_inband_fraction |
| three-figures-render-under-cap _(secondary)_ | kind=size field=figure_html_bytes | op lt value 2500000 field figure_html_bytes |


## Study: Substrate behavior at scale: distance-to-CPM, and how the vertex mechanics were matched (`substrate-behavior-at-scale`)

**Question.** On the LARGE Sego 2022 blueprint (120x120, ~225-cell epithelium), how far does each mechanical substrate's emergent MOVEMENT and INTERACTION pattern sit from the CPM reference, measured by the cross-paradigm behavior metrics (per-cell displacement, MSD, neighbour-graph churn, infection-front speed, cell sorting)? Each metric is a DISTANCE from the CPM; the same-backend seed-to-seed distance is the null floor a match must approach.

**Purpose.** All three backends run the SAME shared mechanism (fields, rules, immune ODE) on the SAME large blueprint; only the mechanics node differs. Differences in the behavior metrics are therefore attributable to the substrate (CPM Metropolis lattice vs vertex confluent polygons vs center-based discs), not the biology.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `sego-large` | `viva_multicell_blueprint.composites.pipeline.blueprint_pipeline` | 0 | backend=cpm, blueprint=workspace/blueprints/influenza_sego2022_large.yaml |
| `neutral-mechanics-sheet` | `viva_multicell_blueprint.composites.pipeline.blueprint_pipeline` | 0 | backend=cpm, blueprint=workspace/blueprints/mechanics_sheet.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.pipeline.blueprint_pipeline`**

Backend-neutral blueprint compiled onto a mechanics backend (cpm|vertex|munk): fields/rules/ODE/immune mechanism, plus an optional per-cell subcellular layer arbitrated into the mechanics inputs by the actuation merge.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm', 'vertex', or 'munk'
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # a path to a YAML file, inline YAML text, or a dict (v1 flat or v2 sectioned)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: substrate-behavior-at-scale ===
STUDY = 'substrate-behavior-at-scale'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_sego_large = 100
INTERVAL_sego_large = 1.0
STEPS_neutral_mechanics_sheet = 100
INTERVAL_neutral_mechanics_sheet = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline if 'ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline = materialize_composite('viva_multicell_blueprint.composites.pipeline.blueprint_pipeline', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline['state']}, core=core)
            _comp.run(STEPS_sego_large)
        print('ran', 'sego-large', f'(viva_multicell_blueprint.composites.pipeline.blueprint_pipeline) for', STEPS_sego_large, 'steps')
    except Exception as _e:
        print(f"sego-large not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline if 'ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline = materialize_composite('viva_multicell_blueprint.composites.pipeline.blueprint_pipeline', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline['state']}, core=core)
            _comp.run(STEPS_neutral_mechanics_sheet)
        print('ran', 'neutral-mechanics-sheet', f'(viva_multicell_blueprint.composites.pipeline.blueprint_pipeline) for', STEPS_neutral_mechanics_sheet, 'steps')
    except Exception as _e:
        print(f"neutral-mechanics-sheet not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Spatial playback — CPM on the large Sego tissue**


In [ ]:
# Spatial playback — CPM on the large Sego tissue
show_viz(_render_one('html:spatial-playback-cpm-large.html', {'title': 'Large Sego 2022 — CPM reference playback (120x120, ~225-cell epithelium)'}, RUNS_DB, STUDY_YAML))

**Vertex mechanics calibration**


In [ ]:
# Vertex mechanics calibration
show_viz(_render_one('html:mechanics-calibration.html', {'title': 'Vertex knobs fitted to CPM area, wandering and chemotactic drift'}, RUNS_DB, STUDY_YAML))

**J ↔ tension sweep**


In [ ]:
# J ↔ tension sweep
show_viz(_render_one('html:adhesion-sweep.html', {'title': 'Cell-shape statistics across the vertex tension sweep vs CPM'}, RUNS_DB, STUDY_YAML))

**Neutral sheet playback — CPM vs vertex vs center-based**


In [ ]:
# Neutral sheet playback — CPM vs vertex vs center-based
show_viz(_render_one('html:neutral-sheet-playback.html', {'title': 'Mechanics-only sheet relaxation, all three paradigms (center-based not calibrated)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| munk-moves-and-interacts _(primary)_ | kind=distance field=per_cell_displacement_distance backend=munk | op gt value 0.5 note munk displacement distance >> 0 (was ~0 when frozen) |
| vertex-closest-to-cpm _(primary)_ | kind=comparison field=msd_distance | op lt note vertex_vs_cpm < munk_vs_cpm on displacement and MSD |
| distances-above-null-floor _(primary)_ | kind=null_control field=all_metrics | op reported note cross vs null floor reported per metric |


## Study: `dose-response-crossparadigm`

**Question.** Do the two substrates agree on the SIGN of the dose response over initial viral load (more load -> more infection), robustly across seeds?

**Purpose.** More initially-infected cells secrete more virus into the shared field, raising the per-cell infection probability (Pr = 1 - exp(-g_hv * virus)) everywhere, so the final infected count rises with the initial dose on either substrate.

**Claim.** RE-RUN 2026-11-05 on the in-sheet immune vertex substrate (re-calibrated motility): figure regenerated; the STRICT-DIRECTION behavior test (highest dose ends with more infection than the lowest, on both backends) still passes. Earlier reading:   Both backends agree on the DIRECTION of the dose response: a HIGHER initial viral load (more initially-infected epithelial cells) ends with STRICTLY more infection than a LOWER load, on every backend and seed tested. This is a directional (lowest-vs-highest) result; intermediate-dose monotonicity is seed-dependent and is NOT claimed.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `sego-dose-sweep` | `viva_multicell_blueprint.composites.pipeline.blueprint_pipeline` | 0 | backend=cpm, blueprint=workspace/blueprints/influenza_sego2022.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.pipeline.blueprint_pipeline`**

Backend-neutral blueprint compiled onto a mechanics backend (cpm|vertex|munk): fields/rules/ODE/immune mechanism, plus an optional per-cell subcellular layer arbitrated into the mechanics inputs by the actuation merge.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm', 'vertex', or 'munk'
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # a path to a YAML file, inline YAML text, or a dict (v1 flat or v2 sectioned)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: dose-response-crossparadigm ===
STUDY = 'dose-response-crossparadigm'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_sego_dose_sweep = 100
INTERVAL_sego_dose_sweep = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline if 'ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline = materialize_composite('viva_multicell_blueprint.composites.pipeline.blueprint_pipeline', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline['state']}, core=core)
            _comp.run(STEPS_sego_dose_sweep)
        print('ran', 'sego-dose-sweep', f'(viva_multicell_blueprint.composites.pipeline.blueprint_pipeline) for', STEPS_sego_dose_sweep, 'steps')
    except Exception as _e:
        print(f"sego-dose-sweep not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Dose response — initial load → final infection**


In [ ]:
# Dose response — initial load → final infection
show_viz(_render_one('html:dose-response.html', {'title': 'Final infected count vs initial dose, both backends and seeds'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| strict-direction-both-backends-and-seeds _(primary)_ | kind=comparison field=final_infected_count low=0.03 high=0.15 | op strictly_greater note final(0.15) > final(0.03) for cpm/vertex/munk x seeds {0,1} |
| higher-dose-seeds-more-cells _(primary)_ | kind=comparison field=seed_infected low=0.03 high=0.15 | op strictly_greater value 5 > 1 |


## Study: Subcellular ODE layer: cells grow and divide on every backend (`subcellular-layer-grow-divide`)

**Question.** Does the pluggable per-cell subcellular (tellurium ODE) layer grow AND divide cells uniformly across cpm, vertex and munk through `blueprint_pipeline`?

**Purpose.** The blueprint's `subcellular` section is compiled into a per-cell ODE step that is independent of the mechanics backend: it reads fields at each centroid, advances S, and writes target_size and division requests to shared stores. Each backend implements the same actuation contract (a target_size change, and a division event mapped onto the backend's own division primitive), so the same blueprint should grow and divide cells on all three substrates. Differences are expected only in how a backend realizes the division event (e.g. CPM has no per-id split).

**Claim.** The pluggable per-cell subcellular layer (a tellurium ODE bound to a cell type in the v2 blueprint) grows AND divides cells uniformly across CPM, vertex and center-based through the single `blueprint_pipeline`. In the demo blueprint a nutrient field sampled at each cell's centroid drives a per-cell growth ODE; the ODE species S actuates target_size (cells grow) and crossing the divide threshold triggers division with a lineage entry on every backend. Cell-count progressions in the 12-step e2e run: CPM 4 -> 8 -> 16, vertex 4 -> 8 -> 16, center-based 4 -> 6 -> 8 -> 16. This is a demonstration of the wiring (blueprint -> pipeline -> per-cell ODE -> backend actuation), not a biological result.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `subcellular-ode-demo` | `viva_multicell_blueprint.composites.pipeline.blueprint_pipeline` | 0 | backend=cpm, blueprint=workspace/blueprints/subcellular_ode_demo.yaml |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_multicell_blueprint.composites.pipeline.blueprint_pipeline`**

Backend-neutral blueprint compiled onto a mechanics backend (cpm|vertex|munk): fields/rules/ODE/immune mechanism, plus an optional per-cell subcellular layer arbitrated into the mechanics inputs by the actuation merge.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline = {
    'backend'  : 'cpm',   # mechanics substrate: 'cpm', 'vertex', or 'munk'
    'blueprint': '/home/runner/work/viva-multicell-blueprint/viva-multicell-blueprint/workspace/blueprints/influenza_sego2022.yaml',   # a path to a YAML file, inline YAML text, or a dict (v1 flat or v2 sectioned)
    'video_out': '',   # if set, record a spatial-playback video to reports/assets/_studio_runs/<video_out>.html
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: subcellular-layer-grow-divide ===
STUDY = 'subcellular-layer-grow-divide'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_subcellular_ode_demo = 100
INTERVAL_subcellular_ode_demo = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline if 'ov_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline' in dir() else None  # edits from the panel cell, if any
            spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline = materialize_composite('viva_multicell_blueprint.composites.pipeline.blueprint_pipeline', _ov)
            _comp = Composite({'state': spec_viva_multicell_blueprint_composites_pipeline_blueprint_pipeline['state']}, core=core)
            _comp.run(STEPS_subcellular_ode_demo)
        print('ran', 'subcellular-ode-demo', f'(viva_multicell_blueprint.composites.pipeline.blueprint_pipeline) for', STEPS_subcellular_ode_demo, 'steps')
    except Exception as _e:
        print(f"subcellular-ode-demo not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| ode-drives-growth _(primary)_ | kind=comparison field=sub_scalar_S_and_sizes | op changes_and_grows note tests/test_subcellular_demo_e2e.py::test_ode_drives_growth[cpm|vertex|munk] |
| ode-drives-division _(primary)_ | kind=count field=cell_count | op gt value 4 field max_cell_count note tests/test_subcellular_demo_e2e.py::test_ode_drives_division[cpm|vertex|munk]; lineage non-empty |
| one-threshold-crossing-one-division _(primary)_ | kind=count field=cell_count when=tick-7 | op eq value 8 field cell_count note tests/test_subcellular_demo_e2e.py::test_one_threshold_crossing_one_division[cpm|vertex|munk] |


## Open decisions
- STAGE 1 (BP-01): formalize a versioned blueprint schema + canonical IR, separate from backend geometric init (units, coord frame, boundaries, tessellation-independent IDs, footprints, fields, contacts, latent states, seeds, init order, mechanism bindings, provenance, allowable loss). The versioned schema, canonical/backend split and static validator exist; the remaining fields are open.
- STAGE 2 (BP-02/BP-04, extend blueprint-pipeline-fidelity): prove blueprint fidelity beyond centroids/IDs -- area, footprint, contact, adjacency, boundary, topology, round-trip, with explicit error measures and loss reports.
- STAGE 3 (BP-07, substrate-behavior-at-scale): the vertex tension scale, immune motility and chemotaxis are calibrated to CPM observables under a no-fold constraint. Open: the vertex epithelium cannot reach CPM fluidity without folding (MSD ~2.2 vs 9.5 um^2) -- push knobs under a mesh-validity gate, add a T1-safe fluctuation term, or accept the residual (VCF-01); area dispersion unmatched; tension scale geometry-dependent (1.74 vs ~4); one cell size.
- STAGE 4 (BP-05): V/L/C/F inputs have no producer and immune recruitment is a latent-pool approximation -- build a producer/consumer ledger, implement or clearly designate deviations, and ablate frozen inputs.
- STAGE 4 (BP-06): the +/-0.05 fate-equivalence margin is NOT yet biologically justified; the dependence-aware reanalysis is done (cluster bootstrap) but power under biologically meaningful perturbations is not established.
- STAGE 4 (BP-08): biological fidelity to Sego 2022 is NOT validated -- digitize figure/table/parameter anchors and compare predictive intervals on held-out data; no biology inferred from cross-backend agreement.
- STAGE 4 (dose-response redefine): separate virion inoculum from seeded infected fraction; 6-10 graded doses x replicate seeds; it is currently directional-only, NOT a dose-response curve.
- SUBCELLULAR LAYER: commit a run (runs.db) with a figure for subcellular-layer-grow-divide and test lineage / size-distribution equivalence across backends over several seeds; give viva-cpm a per-id split or document its divide-all-over-threshold semantics.
- STAGE 5 (BP-09): compile a SECOND independent tissue blueprint through the same schema with no influenza-specific adaptations -- the decisive generality test.
